> # Pre-Lab Instructions
> <img src="https://github.com/Minyall/sc207_290_public/blob/main/images/attention.webp?raw=true" height=200>

> For this lab you will need:
> - DATA: `farright_dataset_cleaned.parquet`. Download from Moodle and upload to this Colab session.
> - INSTALL: You will need to install `networkx` and `python-louvain`.

In [ ]:
#*
# If you're using Colab you will need to run this cell to avoid environment issues
!pip uninstall networkx --yes
!pip uninstall python-louvain --yes
!pip uninstall community --yes
!pip install python-louvain
!pip install networkx

# SC290: Network Analysis - Influence and Power
<img src="https://github.com/Minyall/sc207_290_public/blob/main/images/sc290_headers/8.png?raw=true" height=300 align="right">

## Last week
- Network Basics
- How to visualise a Network in Gephi
- How to use size, color and layout to assist analysis
- How to filter networks to improve analysis


## This week
- One approach to making news article data 'relational'.
- Named entity recognition
- How to transform that data into a network
- How to filter, calculate different metrics of influence and identify communities entirely from Python.
- How to export this to a Gephi compatible file.

# From Text to Network
- Last session we looked at how tv-shows and films could be transformed into relational data by thinking about them differently. 
- By representing them as characters co-occuring in scenes we were able to create a representation that showed us who the most central and critical characters were, and in some cases, demarcate out storylines through the clustering of those characters.
- If you knew the franchise or series, the results would not have been a revelation, and this is a good thing. It means that the way we've represented the data aligns with some underlying truth about the relations between characters.
- Even if you'd never heard of 'Friends' if you were shown that network you would know who the six main characters were. You would perhaps know who the most prominent side characters were, and so forth. 
- Even if you knew the series well, it may also show you patterns you didn't necessarily realise by consuming it directly.

Today we're going to build a dataset for a different series, the news. We'll identify the main characters, the side characters, the storylines and perhaps see things we wouldn't see otherwise.



In [ ]:
#*
# Loading our data and a spacy model
import spacy
import pandas as pd

# Define the nlp model
nlp = spacy.load('en_core_web_sm')

# Load in our dataset
articles = pd.read_parquet('farright_dataset_cleaned.parquet')
articles.info()


In [ ]:
#*
# Take a sample of the first 100 articles
articles = articles.head(100)
articles.info()

## Creating our 'Scenes'
In our TV/film data, the material had been subdivided up into scenes so we could then ask "how often were these characters 'together'". In our data we have articles which can be subdivided into paragraphs and so:
- Think of an article as an episode.
- Think of a paragraph as a scene
If two names are mentioned in a single paragraph it is probably because something relates them together. They are co-occuring in the 'scene'.

First we'll split our articles into paragraphs.

In [ ]:
# We're going to make a column of paragraphs by splitting the cleaned text wherever there is a newline character \n
articles['paragraph'] = articles['cleaned_text'].str.split('\n')

# We can see what the first row of this column looks like as an example
articles.loc[0,'paragraph']


In [ ]:
# Now we're going to make it so that every row is a single paragraph. 
# The index will keep track of which article each paragraph relates to.
paragraph_data = articles.explode('paragraph')

# and then we'll take just the paragraphs column, give it a new index 
# and keep the old one, calling it 'article_idx
paragraph_data = paragraph_data[['paragraph']].reset_index(drop=False, names='article_idx')
paragraph_data

## Getting our 'characters'
We are now going to check each paragraph in each story for the presence of person and organisation names. We refer to these as 'entities' and the process of finding them as... 


### Named Entity Recognition
A technique used in natural language processing to identify specific types of information in text, such as names, organisations, places, dates, etc. In the good old days, entities were identified by matching against great big lists of notable names, places etc. It was developed primarily to improve the ability of computers to answer questions more directly for things like search and building knowledge bases. 

Today transformer based text models like Spacy identify entities based on analysis of the text itself, based on the linguistic relationships between the words used.

> More recently I and partners have found a lot of success using LLM's for named entity recognition. However the process requires patience and careful data management as the process has to be broken down into multiple requests over several hours.


In [ ]:
#*
# An example of different types of entities in a piece of text.

spacy_processed_text = nlp("""UK Health Secretary Wes Streeting said: "I trust doctors over President Trump, frankly, on this."
Mel Merritt, head of policy and campaigns at the National Autistic Society said: "This is dangerous, it's anti-science and it's irresponsible.
"President Donald Trump is peddling the worst myths of recent decades. Such dangerous pseudo-science is putting pregnant women and children at risk and devaluing autistic people.""")

# Source: https://www.bbc.co.uk/news/articles/cdx2rk10ep0o


spacy.displacy.render(spacy_processed_text, style='ent', jupyter=True)

In [ ]:
# To extract the entities in a spacy processed document we access the .ents method

spacy_processed_text.ents

In [ ]:
# and each entity has its own attributes, .text and .label_
[(entity.text, entity.label_) for entity in spacy_processed_text.ents]

In [ ]:
# Let's apply this to our paragraphs

# we define the entity types we want to keep - 
# (this could be a list but generally if you don't intend to change that list it is clearer to use a tuple)
KEEP_ENTS = ('PERSON','ORG')

# Our sample is only 100 items big but if you want to do larger amounts it's good to manage the pipe batch size.
#  Refer back to session 2 for more detail.
BATCH_SIZE = 150

# Our destination list. Each item will be a list of entities in a paragraph
paragraph_entity_lists = []

# For every paragraph in our paragraph_data['paragraph'] colum, processed in a spacy pipe using 1 process, handling 150 items at a time
for para in nlp.pipe(paragraph_data['paragraph'], n_process=1, batch_size=BATCH_SIZE): 

     # We create a list of all the paragraph's entities filtering to check the label is in
     #  our kept entities and the first letter of the entity is uppercase (i.e. a name)
     entities = [e.text for e in para.ents if e.label_ in KEEP_ENTS and e.text[0].isupper()]


     # Finally we make sure that there aren't any duplicates in our list (for example if a name was mentioned twice)
     # A set is like a list but every item must be unique. 
     # You put a list in a set and it drops duplicates
     # You put a set in a list and it turns back into a list again.
     entities = list(set(entities))

     # And send our list of entities to our destination list
     paragraph_entity_lists.append(entities)


# assert checks if a statement is true and throws an error if it is False
# Here we assert that our destination list is the same length as our number of paragraphs
assert len(paragraph_entity_lists) == len(paragraph_data)

# We then take that list and turn it into a column in our dataset
paragraph_data['entities'] = paragraph_entity_lists
paragraph_data



In [ ]:
#*
# Whilst it seemed a lot, without the comments it is quite a concise process.

KEEP_ENTS = ('PERSON','ORG')
BATCH_SIZE = 150

paragraph_entity_lists = []

for para in nlp.pipe(paragraph_data['paragraph'], n_process=1, batch_size=BATCH_SIZE): 
    entities = [e.text for e in para.ents if e.label_ in KEEP_ENTS and e.text[0].isupper()]
    entities = list(set(entities))
    paragraph_entity_lists.append(entities)

assert len(paragraph_entity_lists) == len(paragraph_data)
paragraph_data['entities'] = paragraph_entity_lists
paragraph_data

## From Entities to Network
We now have our list of paragraphs (scenes) and entities (characters). Finally we're going to make a network.
- A Node: Represents an entity in the paragraphs.
- An Edge: Represents a co-occurence of two entities.

Node Attributes
- `n_paragraphs`: Number of paragraphs the entity occurs in.
- `n_articles`: Number of articles the entity occurs in.

Edge Attributes
- `'weight'`: Number of times the two entities co-occur in a paragraph

In [ ]:
# We transform our data to be an entity per row. Again the index tracks which entities were originally together.
row_per_entity = paragraph_data.explode('entities')
row_per_entity

In [ ]:
# Node attributes first
## If we group the above by entities then we can count the number of rows containing that entity, that's n_paragraphs.
# We can also count the number of unique article_idx numbers, that's n_articles

node_attributes = row_per_entity.groupby('entities').agg(
    n_paragraphs=('article_idx', 'count'),
    n_articles=('article_idx','nunique')
    )

# Finally we transform this into a dictionary, ensuring that the key is the entity name (orient='index')
# We'll use this dictionary of attributes later.
node_attr_dict = node_attributes.to_dict(orient='index')
node_attr_dict

In [ ]:
# Next we're going to create an adjacency matrix. 
# This shows the structure of the network and takes care of the edge weight attribute

# First we turn our column of individual entities into a dummy matrix
# Each row represents an entity - for each row every column will be 0 apart from the column matching that entity's name which will be 1.
dummies = pd.get_dummies(row_per_entity['entities'], dtype=int)
dummies

In [ ]:
# To make it one row per paragraph, we group by the index (level=0) and add the rows together.
# Now each row shows 1 in the column if that entity is in the paragraph, otherwise 0.
dummies_per_para = dummies.groupby(level=0).sum()
dummies_per_para

In [ ]:
# Magic time
# This is called matrix multiplication. It is a key part of graph theory.
# All we need to know is that it is able to calculate how many times each of the items in our columns co-occur.

# An adjacency matrix has all the information needed to make a weighted network

adjacency_matrix = dummies_per_para.T.dot(dummies_per_para)
adjacency_matrix

In [ ]:
#*
# As an example we can check one entity and see what other entites co-occur with them.
adjacency_matrix.loc['Keir Starmer'].sort_values(ascending=False)

Note that the entity will always have a co-occurence with itself. This number represents the number of times the entity occurs, in our case the number of paragraphs it shows up in.

In [ ]:
#*
node_attributes.loc['Keir Starmer']

The adjacency matrix is a tabular way (i.e. table shaped) of representing edges in a network, with the values representing the weights of those edges. So we have:
- An adjacency matrix representing our edges
- A dictionary of node attributes, representing our nodes

In effect, we have our network, what matters now is structuring it as a network so we can treat it as one, and analyse it as one. Enter `NetworkX`.

# NetworkX

In [ ]:
# Finally we can make our network using networkx

import networkx as nx

G = nx.from_pandas_adjacency(adjacency_matrix)
nx.set_node_attributes(G, node_attr_dict)

print(G.number_of_nodes(), G.number_of_edges())


In [ ]:

# We can check if the network's node attributes look right using...
G.nodes(data=True)


In [ ]:
# and the edges using 
G.edges(data=True)

## Filtering with NetworkX
Whilst you can filter with Gephi, you may find it useful to filter with networkx for various reasons. 
- Primarily it may be that having explored your data in Gephi, you want to lock in place a certain filtered version of your data. 
    - Solidifying your filtering into clear steps in your code makes it clear exactly what filtering took place, in what order, and ensures it is consistent if you need to run analysis again.
- You may want to use NetworkX to get exact figures on various metrics, and not want to shift over the Gephi to retrieve them.
- You may want to integrate the outputs of network analysis into a larger project - (more on this next week).


Let's apply some filters. If we were to do them in Gephi they'd look at bit like this.

<img src="https://github.com/Minyall/sc207_290_public/blob/main/images/gephi_settings.png?raw=true" height=200>

We're going to do them directly in Python

In [ ]:
#*
# We'll rebuild the graph again just so we're clear on what we're starting with.
G = nx.from_pandas_adjacency(adjacency_matrix)
nx.set_node_attributes(G, node_attr_dict)
print(G.number_of_nodes(), G.number_of_edges())

The two main filtering methods are `.remove_nodes_from` and `.remove_edges_from`. Each takes either a list of nodes to remove, or a list of edges. To filter the graph in some way, what differs is how you identify the nodes and/or edges to remove.

Keep in mind the order of filtering. Each stage of filtering changes the structure of the graph so understanding what happened in each stage is important.

### Filtering Self-Loops
A self-loop is where a node has an edge that connects to itself. In our network this happens because of the way we built our edges using matrix multiplication, where we ended up with entities co-occuring with themselves.

A self-loop still counts as a (very highly weighted) edge and so can throw off any metrics that factor in degree or edge weights.

We'll get rid of them now.

In [ ]:
remove_edges = nx.selfloop_edges(G)
G.remove_edges_from(remove_edges)

### Filter on Edge Attribute (Weight)
This filter removes edges if they have a weight below a certain value. Here we'll go wth filtering edges if they have a weight of 1.

This means we're getting rid of relations between entitities if they only co-occur once.

The result will be less edges. It may also mean we have nodes that have 0 edges.

In [ ]:
# Remove low weight edges
low_weight_edges = [(source,target) for source, target, attr_dict in G.edges(data=True) if attr_dict['weight'] < 2]
G.remove_edges_from(low_weight_edges)

### Filter by Structure (Degree)
Next we remove nodes if they have a degree of 0, essentially if they no longer co-occur with anyone after we deleted those low weight edges.

A more aggressive approach may remove nodes if they only co-occur with 1 other person but for now we'll just remove detached (edgeless) nodes.

In [ ]:

# Remove nodes based on degree
disconnected_nodes = [node for node, degree in G.degree if degree < 1] # if you wanted to filter low degree nodes you would just change the value to 2, 3, 4 etc
G.remove_nodes_from(disconnected_nodes)

### Filter by Structure (Component)

<img src="https://github.com/Minyall/sc207_290_public/blob/main/images/Pseudoforest.png?raw=true" height=200 align="right">

It is likely with our kind of data that there are little islands of entities that co-occur with one another. These little islands aren't noise necessarily. For example if you have a set of stories all revolving around a few key individuals, but none of them are mentioned alongside other more prominent figures they will form a small island or 'component'.

The largest collection of joined up entities is called the 'giant component'. There is good reason NOT to filter out nodes just because they're not part of the giant component, as it can inform you about seperate but still important patterns.

However, having multiple components can majorly interfere with community detection and may get in the way of teaching later so we will filter to just retain the main giant component.

In [ ]:
components = list(nx.connected_components(G))
len(components)


In [ ]:
# We can see what one of the components looks like
components[0]

It is worth checking to see whether there might be a number of large components or just one big one.

In [ ]:
#*
component_sizes = [len(c) for c in components]
import plotly.express as px
px.bar(component_sizes, title='Component Sizes')

In [ ]:

# Get the LARGEST component
largest_component = max(components, key=len)

# If you wanted to retain multiple components you could 
# use the chart above to identify the index positions of the ones you want
# and build a list

component_indexes = (6,15,19)
custom_component = []
for index in component_indexes:
    custom_component.extend(components[index])


In [ ]:

# Subgraph filters a graph based on a set of nodes,
#  and retains the edges between those nodes.

# We'll use the largest component
G = G.subgraph(largest_component)

print(G.number_of_nodes(), G.number_of_edges())

## Community Detection
Whilst we can do community detection in Gephi, we can also do it in NetworkX. Why? 
- Because we may want those community assignments available to us here in Python without stepping over to Gephi.
- We may want to test different settings and see their effects.
- It helps reinforce the understanding that a Network is not just about the visual 'picture' but that it ultimately is a representation of relational data that we can interrogate.

In [ ]:
import community

In [ ]:
communities = community.best_partition(G, random_state=42)
communities

In [ ]:
# Again we can count the number of 
# communities by using a Pandas Series

pd.Series(communities, name='community').nunique()

In [ ]:
# We can calculate the modularity score
# Remember 0.4+ is considered good
community.modularity(communities, G)

In [ ]:
graph_data = node_attributes.merge(pd.Series(communities, name='community'), left_index=True, right_index=True)
graph_data = graph_data.reset_index(names='entity')
graph_data

In [ ]:
community_stats = graph_data.groupby('community').agg(community_size=('n_paragraphs','count'),
                                         para_per_community=('n_paragraphs','sum'),
                                         articles_per_community=('n_articles','sum')).sort_values('community_size',ascending=False)

In [ ]:
top_communities = community_stats.head().index
grouped_community_data = graph_data.groupby('community')
for com in top_communities:
    subset = grouped_community_data.get_group(com)
    top_entities = subset.sort_values('n_articles',ascending=False)['entity'].head()
    print(f"Community Number: {com}")
    for ent in top_entities:
        print(ent)
    print('*'*5)

## Graph Metrics
A key benefit of representing you data as a network is the different kinds of measures that you can derive from that structure.

All the metrics work in a similar way.

In [ ]:
# Get a dictionary of scores for the chosen metric
scores = nx.degree_centrality(G)
scores

In [ ]:
# You can put these scores into a Pandas Series to 
# examine them independently
pd.Series(scores, name='degree_centrality').sort_values(ascending=False)

In [ ]:
# create a new column for the metric and use map to match the scores to the right rows
graph_data['degree_centrality'] = graph_data['entity'].map(scores)
graph_data

In [ ]:
graph_data.sort_values('degree_centrality', ascending=False)

A simple model for this is below, just switch the metric name to the one you want.

In [ ]:
metric_options = {
    'degree_centrality': nx.degree_centrality,
    'eigenvector': nx.eigenvector_centrality,
    'betweenness': nx.betweenness_centrality,
    'closeness':nx.closeness_centrality,
    'page_rank':nx.pagerank
}

CHOSEN_METRIC = 'eigenvector'

scores = metric_options[CHOSEN_METRIC](G)
graph_data[CHOSEN_METRIC] = graph_data['entity'].map(scores)

top_n = 10
to_plot = graph_data.sort_values(CHOSEN_METRIC, ascending=False).head(top_n)
fig = px.bar(data_frame=to_plot, y='entity', x=CHOSEN_METRIC, title=f'Top {top_n} Entities in Network by {CHOSEN_METRIC}')
fig.update_yaxes(categoryorder='total ascending')

In [ ]:
top_communities

In [ ]:
# What if you want to know the scores for nodes within a specific community instead? 
# For all the figures within a specific community, which of them is the most important?
# The metrics calculated above consider the whole network, what if we want just a subset of the network?

CHOSEN_METRIC = 'degree_centrality'
COMMUNITY = 54
subset = graph_data[graph_data['community'] == COMMUNITY].copy()
chosen_nodes = subset['entity']

sub_G = G.subgraph(chosen_nodes)

scores = metric_options[CHOSEN_METRIC](sub_G)
subset[CHOSEN_METRIC] = subset['entity'].map(scores)

top_n = 20
to_plot = subset.sort_values(CHOSEN_METRIC, ascending=False).head(top_n)
fig = px.bar(data_frame=to_plot, y='entity', x=CHOSEN_METRIC, height=600,
              title=f'Top {top_n} Entities in Community {COMMUNITY} by {CHOSEN_METRIC}')
fig.update_yaxes(categoryorder='total ascending')

## Exporting to Gephi
All of the metrics we generate here can also be generated in Gephi, however for simplicty and to ensure consistency across all versions of your data it's simpler to generate them here and embed them in the graph to be used in Gephi if needed.

In [ ]:
# We iterate over each metric and run it, assigning the scores to the network object
for metric_name, metric_function in metric_options.items():
    scores = metric_function(G)
    nx.set_node_attributes(G,scores,metric_name)
G.nodes(data=True)

In [ ]:
# Finally we add the community assignments
nx.set_node_attributes(G, communities, 'community')

# And then export to gephi
nx.write_gexf(G,'my_entity_graph.gexf')

### Notes on Metrics
#### Degree Centrality
We know degree is the number of edges a node has. However, *Degree Centrality* is a slightly adjusted version of this that makes it easier to know if the degree is high or low. Degree centrality is simply a node's degree frequency, divided by the total number of possible connections it could have (not including itself). As such it is a measure showing us the proportion of possible other nodes that a node connects to.

#### Betweenness Centrality
Indicates the extent to which a node stands between two others. In our data this could indicate people central to the issue overall across the different stories, but it could also indicate people that connect a side issue to a larger whole. 

Unweighted just considers if there is an edge, weighted factors in if co-occurences happen a lot.
#### Eigenvector Centrality and PageRank
Indicates the importance of a node based on the importance of the nodes it is connected to. For us this could indicate our core people, but also individuals that may be important by virtue of their closeness to important people.
#### Closeness Centrality
How close is a node to the rest of the network? On average how many steps would it take to get from a node to any other node in the network. A high closeness centrality indicates that a node is closer to all other nodes. It is often used to indicate access to other nodes, or information flow in a network. How easy would it be for you to be introduced to any other person in the university?

In our graph this indicates how connected a person is across a range of topics. If someone could easily 'step' from their position to any other person in the network in small number of steps they are likely an important figure in the issue.